# Project 5: Smart Factory Machine Maintenance Analysis
**Industry:** Manufacturing / Industrial IoT | **Tools:** Python, NumPy, Pandas

Run the cells in order (Runtime → Run all). Findings are associations, not proof of causation.

## Setup: upload the dataset
Run this cell and choose `05_machine_maintenance.csv` when prompted (Colab only). Skip it if the file is already in the session.

In [ ]:
import os
if not os.path.exists('05_machine_maintenance.csv'):
    from google.colab import files
    files.upload()   # select 05_machine_maintenance.csv
print(os.path.exists('05_machine_maintenance.csv'))

## Task 64. Load and inspect

In [ ]:
import numpy as np
import pandas as pd

# Failure_Type uses the text "None" for healthy records; by default pandas would
# treat it as missing (NaN), so only truly empty cells are treated as missing.

df = pd.read_csv("05_machine_maintenance.csv", keep_default_na=False, na_values=[""])
print(df.shape)
print(df.head())
df.info()
print(df.describe())

## Task 65. Missing sensor readings

In [ ]:
SENSORS = ["Temperature_C", "Vibration_mm_s", "Pressure_bar", "Operating_Hours", "Tool_Wear_min"]
print("Missing:\n", df.isna().sum())
# Justification: sensor readings are skewed and machine-dependent, so fill with
# the MEDIAN of the same machine (robust to spikes). Dropping rows would lose
# ~5% of records and could remove maintenance cases.
# (Machine_ID is cleaned first so the groups are correct; see step 66.)

## Task 66. Duplicates and machine identifiers

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Record_ID:", df.duplicated(subset="Record_ID").sum())
df = df.drop_duplicates(subset="Record_ID").reset_index(drop=True)
print("Machine IDs before cleaning:", sorted(df["Machine_ID"].unique()))
df["Machine_ID"] = df["Machine_ID"].str.strip().str.upper()    # fix 'm03' vs 'M03'
print("Machine IDs after cleaning :", sorted(df["Machine_ID"].unique()))

for col in SENSORS:
    df[col] = df[col].fillna(df.groupby("Machine_ID")[col].transform("median"))
print("Missing after imputation:", df[SENSORS].isna().sum().sum())

## Task 67. Mean, median, std

In [ ]:
stats = pd.DataFrame({
    "Mean": df[SENSORS].mean(),
    "Median": df[SENSORS].median(),
    "Std": df[SENSORS].std(),
}).round(2)
print(stats)

## Task 68. Maintenance-required counts and percentages

In [ ]:
counts = df["Maintenance_Required"].value_counts()
pct = df["Maintenance_Required"].value_counts(normalize=True).mul(100).round(1)
print(pd.DataFrame({"Count": counts, "Percent": pct}))

## Task 69. Average temperature by maintenance status

In [ ]:
print(df.groupby("Maintenance_Required")["Temperature_C"].mean().round(2))
# Full maintenance-status comparison across all numeric columns
status_comparison = df.groupby("Maintenance_Required").mean(numeric_only=True).round(2)
print(status_comparison)

## Task 70. Vibration: maintenance-required vs others

In [ ]:
print(df.groupby("Maintenance_Required")["Vibration_mm_s"].agg(["mean", "median", "std"]).round(2))

## Task 71. Rank machines by average operating hours

In [ ]:
rank = df.groupby("Machine_ID")["Operating_Hours"].mean().sort_values(ascending=False).round(0)
print(rank.to_frame("Avg_Operating_Hours").assign(Rank=range(1, len(rank) + 1)))

## Task 72. Unusually high vibration (IQR)

In [ ]:
q1, q3 = df["Vibration_mm_s"].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
df["Vibration_Outlier"] = df["Vibration_mm_s"] > upper
print(f"Upper fence: {upper:.2f} | outliers: {df['Vibration_Outlier'].sum()}")
print(df[df["Vibration_Outlier"]].sort_values("Vibration_mm_s", ascending=False)
      [["Record_ID", "Machine_ID", "Vibration_mm_s", "Maintenance_Required"]].head())

## Task 73. Group by Failure_Type

In [ ]:
print(df["Failure_Type"].value_counts())
print(df.groupby("Failure_Type")[SENSORS].mean().round(2))

## Task 74. Average tool wear by failure type

In [ ]:
print(df.groupby("Failure_Type")["Tool_Wear_min"].mean().sort_values(ascending=False).round(1))

## Task 75. NumPy Boolean indexing: high temperature

In [ ]:
temps = df["Temperature_C"].to_numpy()
HIGH_TEMP = np.percentile(temps, 90)           # top 10% of readings
mask = temps > HIGH_TEMP
print(f"High-temp threshold (P90): {HIGH_TEMP:.1f} C | records: {mask.sum()}")
high_temp_df = df[mask]
print(high_temp_df[["Record_ID", "Machine_ID", "Temperature_C", "Maintenance_Required"]].head())
print("Maintenance rate in high-temp records:",
      round((high_temp_df["Maintenance_Required"] == "Yes").mean() * 100, 1), "%")

## Task 76. Risk band (documented thresholds)

In [ ]:
# Score 1 point for each condition: Temperature > 80 C, Vibration > 5.5 mm/s,
# Tool wear > 140 min. Risk: 0 = Low, 1 = Medium, 2+ = High.
score = ((df["Temperature_C"] > 80).astype(int)
         + (df["Vibration_mm_s"] > 5.5).astype(int)
         + (df["Tool_Wear_min"] > 140).astype(int))
df["Risk_Band"] = np.where(score >= 2, "High", np.where(score == 1, "Medium", "Low"))
print(df["Risk_Band"].value_counts())
print(pd.crosstab(df["Risk_Band"], df["Maintenance_Required"], normalize="index").mul(100).round(1))

## Task 77. Correlations among numeric sensors

In [ ]:
corr = df[SENSORS].corr().round(2)
print(corr)
# Interpretation: Operating_Hours and Tool_Wear_min should be positively related
# (tools wear with use). Correlation does not imply causation.

## Task 78. Recommendations

In [ ]:
worst = df.groupby("Machine_ID")["Maintenance_Required"].apply(lambda s: (s == "Yes").mean()).idxmax()
recs = [
    f"Prioritise inspection of {worst}, the machine with the highest maintenance-required rate.",
    "Trigger alerts when temperature, vibration and tool wear cross the High risk band thresholds.",
    "Replace tools proactively before wear reaches the levels seen in Tool Wear failures.",
    "Add vibration monitoring with IQR-based outlier alerts to catch bearing/alignment faults early.",
    "Fix data quality at source (sensor dropouts, duplicate records, inconsistent machine IDs).",
]
for i, r in enumerate(recs, 1):
    print(f"{i}. {r}")

## Flagged-record list (expected deliverable)

In [ ]:
flagged = df[df["Vibration_Outlier"] | (df["Risk_Band"] == "High") | (df["Temperature_C"] > HIGH_TEMP)].copy()
flagged["Flag_Reason"] = np.select(
    [flagged["Vibration_Outlier"], flagged["Risk_Band"] == "High"],
    ["High vibration (IQR)", "High risk band"],
    default="High temperature (P90)")
flagged = flagged[["Record_ID", "Machine_ID", "Temperature_C", "Vibration_mm_s",
                   "Tool_Wear_min", "Risk_Band", "Maintenance_Required", "Flag_Reason"]]
print("Flagged records:", len(flagged))
print(flagged.head(10))
flagged.to_csv("flagged_records.csv", index=False)

## Save cleaned data

In [ ]:
df.to_csv("machine_cleaned.csv", index=False)

## Download outputs (Colab)
Optional: downloads the flagged-record list and cleaned data.

In [ ]:
from google.colab import files
files.download('flagged_records.csv')
files.download('machine_cleaned.csv')